# 4-2.딥러닝 모델 변환과 양자화

### 학습 목표

- 모델 변환과 양자화의 개념을 이해하고, PyTorch 모델을 다양한 포맷으로 변환할 수 있습니다.
- ONNX와 TensorRT를 활용해 모델을 최적화하고, 추론 속도를 개선할 수 있습니다.
- 다양한 모델 포맷의 특징과 사용 사례를 이해하고 적용할 수 있습니다.
- 최적화된 모델을 실제 환경에 배포하고, 성능을 평가할 수 있습니다.

## 1. 모델 변환

딥러닝 모델을 실제 서비스 환경에 배포할 때는 **모델 크기**, **메모리 사용량**, **추론 속도**가 매우 중요합니다.  
하지만 많은 개발자들은 학습할 때 사용하던 그대로, **PyTorch + Python + Transformers** 조합으로 모델을 바로 서비스에 올리려고 합니다.  
이 방식은 연구·개발 단계에서는 편리하지만, 실제 서비스 환경에서는 여러 이유로 성능이 떨어질 수 있습니다.

### 1.1 왜 PyTorch + Python으로 그대로 배포하면 느릴까?

<img src="image/python_slow.jpg" width="500">

이미지 출처 : https://www.youtube.com/watch?v=bC428xPW1H8

#### 1) Python 인터프리터의 오버헤드
- Python은 인터프리터 언어이기 때문에 실행할 때마다 코드를 해석하며 추가 오버헤드가 발생합니다.
- LLM처럼 토큰을 빠르게 반복 생성해야 하는 상황에서는 이 해석 오버헤드가 **누적되어 느려지는 원인**이 됩니다.

#### 2) PyTorch의 유연성은 큰 장점이지만, 추론에서는 오버헤드가 생길 수 있음
- PyTorch는 연구/개발 단계에서 모델 구조를 빠르게 바꾸고 실험하기 쉬운 **유연성**이 큰 장점입니다.
- 이 유연성 덕분에 다양한 연산을 세밀하게 제어할 수 있지만,
- 추론 단계에서는 연산이 잘게 나뉘어 실행되면서 커널 호출 횟수가 늘어날 수 있습니다.
- 예를 들어 LayerNorm, GELU, matmul이 각각 따로 실행되면 호출 오버헤드가 누적되어 지연시간이 커질 수 있습니다.
- ONNX Runtime, TensorRT 같은 추론 엔진은 이런 연산을 합치고(fusion), 하드웨어에 맞는 커널을 선택해 오버헤드를 줄입니다.


#### 3) Python의 GIL 때문에 "동시에 여러 CPU 작업"이 답답해질 수 있음

- 먼저 **스레드(Thread)** 란, 하나의 프로그램 안에서 여러 작업을 동시에 처리하기 위해 나눈 “작업 단위”입니다.  
  예를 들어, 데이터를 불러오는 작업과 전처리 작업을 동시에 나눠서 처리하고 싶을 때 사용합니다.

- 보통은 스레드를 여러 개 쓰면 CPU 코어를 동시에 활용해서 빨라질 것이라고 생각합니다.

- 하지만 Python에는 **GIL(Global Interpreter Lock)** 이 있어서, 한 순간에 **오직 하나의 스레드만 Python 코드를 실행**할 수 있습니다.

- 그래서 스레드를 여러 개 만들어도 실제 CPU 연산은 번갈아 실행될 뿐, 기대만큼 병렬로 처리되지 않을 수 있습니다.

- 이때 GPU 연산이 빠르더라도, CPU가 데이터 전처리·후처리를 처리하느라 느리면 전체 응답 시간이 늘어납니다.

쉽게 말하면:

- GPU는 계산을 빨리 끝내고 기다리는데  
- CPU는 작업을 하나씩 처리하느라 밀려 있고  
- 결국 전체 파이프라인 속도가 떨어지는 상황입니다.

### 1.2 모델 변환과 다양한 포맷의 필요성

앞에서(1.1) 살펴보았듯, PyTorch + Python 기반의 기본 구조는  
서비스 환경에서 **추론 속도**, **메모리 효율**, **CPU/GPU 활용도** 측면에서 한계가 있습니다.  
이러한 문제를 해결하기 위해 모델을 **특정 하드웨어와 환경에 최적화된 포맷으로 변환**하는 과정이 필요합니다.

<img src="image/model_convert.jpg" width="500">

이미지 출처 : http://learnopencv.com/how-to-convert-a-model-from-pytorch-to-tensorrt-and-speed-up-inference/

모델 변환을 통해 다음과 같은 최적화가 가능해집니다:

- 불필요한 연산 제거 및 그래프 구조 단순화  
- 연산 간 커널 합치기(fusion)로 실행 오버헤드 감소  
- FP16/INT8 같은 정밀도 감소 기반 최적화  
- CPU/GPU의 하드웨어 특성을 활용한 고속 추론  

이제 실제 서비스 환경에서 자주 사용되는 주요 모델 포맷들을 살펴보겠습니다.

## 2. 다양한 모델 포맷과 목적

### 2.1 ONNX (Open Neural Network Exchange)

ONNX는 딥러닝 모델을 **어떤 프레임워크(PyTorch, TensorFlow 등)에서 만들었는지와 상관없이**,  
모두가 공통으로 읽을 수 있는 **표준 모델 형식**으로 변환해 주는 포맷입니다.

쉽게 말하면:

- PyTorch 모델 → 자기만의 방식  
- TensorFlow 모델 → 또 다른 방식  
- 각자 “방언”을 쓰듯이 내부 구조가 모두 다름  

ONNX는 이 모델들을 **전 세계 어디서나 통하는 공통 언어**로 번역해 주는 역할을 합니다.

이렇게 통일된 규격으로 변환하면 다음과 같은 장점이 생깁니다:

- Python을 거치지 않고도 실행 가능 → 불필요한 오버헤드 제거  
- 다양한 엔진(ONNX Runtime, TensorRT 등)에서 그대로 실행 가능 → 호환성 높음  
- 실행 전에 **그래프 최적화**, **커널 합치기(fusion)** 등을 자동으로 적용하여 더 빠르게 실행됨  

즉, ONNX는 “프레임워크를 넘나들 수 있는 표준 모델 규격”이며,  
모델을 더 빠르고 효율적으로 실행하기 위한 첫 단계로 자주 사용됩니다.

### 2.2 TensorRT — NVIDIA GPU에서 매우 높은 성능을 내기 쉬운 방법

TensorRT는 NVIDIA에서 만든 **GPU 전용 추론 엔진**입니다.  
일반적으로 NVIDIA GPU 환경에서 낮은 지연시간과 높은 처리량을 내기 좋습니다.

그런데 TensorRT가 왜 빠른 경우가 많을까요? 주요 이유는 다음과 같습니다:


#### 1) 모델을 하드웨어 맞춤형으로 다시 조립해줌
- 일반 모델은 GPU가 가장 효율적으로 처리하는 형태로 만들어져 있지 않을 수 있음  
- TensorRT는 모델 내부 연산을 분석해서 **GPU가 가장 좋아하는 방식으로 구조를 재배치**함  
  (예: 여러 연산을 하나로 합치기 → 연산 횟수 감소)

#### 2) 정밀도를 낮춰 속도를 크게 올림 (FP16 / INT8)
- 원래 모델은 “FP32”라는 높은 정밀도로 계산함  
- 하지만 실서비스에서는 이렇게 높은 정밀도가 항상 필요하지 않음  
- TensorRT는 수학적 정밀도를 **절반(FP16)** 또는 **더 낮게(INT8)** 줄여  
  **속도는 2~10배 빨라지고, 메모리 사용량도 줄어듦**


#### 3) GPU에 맞춘 최적화된 커널을 자동 선택
- GPU는 내부적으로 수많은 연산 커널을 가지고 있음  
- TensorRT는 이 중 **현재 모델과 GPU에 가장 잘 맞는 커널 조합**을 알아서 고름  
- 사용자가 직접 설정할 필요 없음



#### 4) ONNX 모델을 받아서 “고속 버전”으로 다시 빌드
- 대부분의 경우 PyTorch → ONNX → TensorRT 순서로 변환함  
- ONNX 모델을 입력으로 받아 **TensorRT 엔진 파일(.engine)** 로 변환  
- 이 파일은 GPU에서 바로 실행되며 **가장 빠른 형태의 최적화 모델**이 됨


#### 요약

TensorRT는 다음과 같은 상황에서 가장 큰 효과를 줍니다:

- **NVIDIA GPU에서 초고속 추론이 필요할 때 (NVIDIA가 아닌 하드웨어에서는 작동하지 않음)**  
- **대규모 LLM, Vision 모델 등 계산량이 많은 모델을 서비스할 때**  
- **지연시간(응답속도)이 매우 중요한 실시간 서비스** (챗봇, 스트리밍, 영상 처리 등)

즉, TensorRT는 “NVIDIA GPU 전용 터보 부스터” 같은 역할을 하며,  
모델을 **더 가볍고 더 빠르고 더 효율적으로** 실행할 수 있게 해준다.


### 2.3 LLM 특화 엔진 (vLLM / TensorRT-LLM)

LLM 특화 엔진은 Llama, Gemma 같은 대규모 언어 모델을  
**일반 PyTorch보다 더 빠르고 효율적으로 실행하기 위해 설계된 전용 추론 엔진**입니다.

일반 구현은 연구/실험에는 좋지만, 실제 서비스에서는
- 메모리 낭비
- 요청 동시 처리 한계
- 토큰 생성 지연
이 쉽게 발생합니다.

vLLM, TensorRT-LLM은 이 문제를 줄이기 위해
**메모리 관리 방식**과 **요청 스케줄링 방식**, **커널 실행 방식**을 LLM에 맞게 바꾼 엔진입니다.


#### 왜 LLM 특화 엔진이 빠를까?

#### 1) 왜 하필 KV를 저장할까? (KV 캐시의 기초)
어텐션(Attention)에서 자주 나오는 Q/K/V를 먼저 역할로 보면 이해가 쉽습니다.
- **Q(Query)**: 지금 만들려는 다음 단어의 질문
- **K(Key)**: 과거 단어들의 색인(찾기 위한 키)
- **V(Value)**: 과거 단어들의 실제 내용

LLM은 토큰을 1개씩 생성합니다.
이때 새 토큰의 Q는 새로 계산하면 되지만, 과거 토큰의 K/V는 매번 다시 계산할 필요가 없습니다.

문제는 기본 방식으로 구현하면 과거 K/V를 반복 계산하거나 불필요 복사가 생겨,
문장이 길어질수록 지연시간이 커진다는 점입니다.

그래서 한 번 계산한 과거 K/V를 GPU 메모리에 저장해 재사용하는데,
이것이 **KV 캐시**입니다.


#### 2) vLLM의 핵심: PagedAttention (메모리 관리)

KV 캐시를 어떻게 저장하느냐가 성능을 크게 좌우합니다.

- **기존 방식(정적 크게 할당)**
  - 앞으로 길어질 가능성을 고려해 큰 메모리를 미리 잡아둡니다.
  - 실제로 짧게 끝난 요청에서는 예약 메모리 대부분이 낭비됩니다.

- **vLLM 방식(PagedAttention)**
  - KV 캐시를 작은 페이지(블록) 단위로 나눠 관리합니다.
  - 필요한 만큼만 할당하고, 메모리 빈 공간을 유연하게 활용합니다.

결과적으로 메모리 낭비와 단편화가 줄어들고,
동시에 더 많은 요청을 안정적으로 처리할 수 있습니다.


#### 3) Continuous Batching + LLM 전용 고성능 커널

여러 사용자를 동시에 처리할 때, LLM 특화 엔진은 GPU를 더 바쁘게 유지합니다.

- **Static batching(기존)**
  - 한 배치를 묶어 실행한 뒤, 가장 느린 요청이 끝날 때까지 대기하는 구간이 생기기 쉽습니다.  
  - >배치 안에서 먼저 끝난 요청이 있어도, 가장 오래 걸리는 요청이 끝날 때까지 함께 묶여 있어 GPU 자원이 비효율적으로 사용될 수 있습니다.
- **Continuous batching(vLLM 등)**
  - 토큰 생성 매 반복마다 스케줄링을 갱신합니다.
  - 먼저 끝난 요청은 즉시 빠지고, 대기 중인 새 요청이 바로 들어옵니다.
  - GPU 유휴 시간이 줄어 TPS(초당 토큰 처리량)가 개선됩니다.

여기에 **LLM 전용 fused kernel(연산 융합)** 이 더해집니다.
- 여러 연산을 하나의 커널로 묶어 메모리 왕복을 줄이고
- 실제 토큰 생성 속도를 추가로 끌어올립니다.


##### ✔ 어떤 상황에서 많이 쓰일까?

- 챗봇/질문응답 API
- 문서 요약/번역 서비스
- 동시 요청이 많은 LLM 서버

##### ✔ 요약

LLM 특화 엔진의 핵심은 아래 3가지입니다.
- **KV 캐시를 효율적으로 저장(PagedAttention)**
- **요청을 끊김 없이 교체(Continuous Batching)**
- **연산을 융합해 실행(Fused Kernel)**

즉, "LLM 서비스에서 실제 병목이 생기는 지점"을 정확히 겨냥해,
같은 모델이라도 더 빠르고 안정적으로 운영할 수 있게 해주는 엔진입니다.


### 2.4 모바일·엣지 전용 포맷 (CoreML / TensorFlow Lite)

모바일·엣지 전용 포맷은 iPhone, Android, IoT 기기처럼  
**서버가 아닌 로컬 디바이스에서 모델을 직접 실행**하도록 최적화된 형태의 모델 포맷입니다.  
서버에 요청을 보내지 않고도 기기 안에서 바로 추론이 가능하기 때문에  
빠른 응답 속도와 낮은 전력 소비가 중요한 환경에서 많이 사용됩니다.

대표적인 포맷으로는 **CoreML(Apple)**, **TensorFlow Lite(Android/IoT)** 가 있습니다.

#### 어떤 최적화가 적용될까?

- **모델 경량화(Compression)**  
  모바일 기기가 가진 제한된 메모리·저장 공간에 맞게 모델 크기를 줄입니다.

- **정밀도 감소(Quantization)**  
  FP32 대신 INT8, FP16 등을 사용해 속도를 높이고 배터리 소비를 줄일 수 있습니다.

- **모바일 하드웨어 가속기 활용**  
  iPhone의 Neural Engine, Android의 DSP/NPU 등  
  각 기기 내부의 전용 AI 가속기를 활용해 추론 속도를 높입니다.

#### 어떤 상황에서 적합할까?

- 인터넷 연결이 불안정한 환경  
- 실시간 응답이 중요한 앱(카메라 필터, 음성 인식 등)  
- 배터리 절약이 중요한 휴대기기  
- IoT 센서·스마트기기에서 로컬 추론이 필요한 경우

#### 요약

모바일·엣지 전용 포맷(CoreML / TFLite)은  
로컬 디바이스에서 **가볍고 빠르게 추론**하기 위해 최적화된 모델 형식입니다.  
서버 비용을 줄이고, 빠른 반응 속도를 제공해야 하는 모바일·IoT 환경에서 매우 유용합니다.


#### 정리

- 연구·학습단계에서 사용하던 PyTorch + Python 구조는 **추론 환경에서는 비효율적**  
- 환경에 맞는 포맷으로 변환하면 **속도, 메모리, 하드웨어 효율**을 크게 개선할 수 있음  
- 서버 환경이면 ONNX/TensorRT, LLM 환경이면 vLLM·TensorRT-LLM,  
  모바일 환경이면 CoreML/TFLite처럼 **상황에 맞는 최적 포맷을 택하는 것이 핵심**



### 2.5 실습 : PyTorch 모델을 ONNX로 변환

이번 단계에서는 PyTorch 모델을 **ONNX 포맷으로 변환하는 실습**을 진행합니다.  
ONNX는 다양한 실행 엔진(ONNX Runtime, TensorRT 등)에서 공통으로 사용할 수 있는 표준 포맷으로,  
모델을 다른 환경에서 빠르게 실행하거나 최적화할 때 자주 사용됩니다.

여기서는 **ResNet18 (이미지 분류 모델, CNN)** 모델을 대상으로 변환을 진행합니다.  
변환 과정과 속도가 어떻게 달라지는지 확인하는 것이 목표입니다.

#### 코드 예시: ResNet18
아래 코드는 다음 과정을 모두 포함합니다.

- PyTorch 모델 로드
- ONNX 변환 + 변환 시간 측정
- 실제 이미지 파일(`image/dog.jpg`) 불러오기
- PyTorch 추론 시간 측정
- ONNX Runtime 추론 시간 측정
- 두 결과 비교

성능 측정에서 꼭 기억할 점:
- 단일 1회 측정은 편차가 큽니다.
- 특히 batch=1에서는 데이터 복사/런타임 초기화/I/O 오버헤드 비중이 커집니다.
- 따라서 **워밍업 후 반복 측정(평균/표준편차)** 으로 비교해야 해석이 안정적입니다.


In [1]:
import time
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import torch.onnx
import onnxruntime as ort

# 1. 모델 준비 (ResNet18)
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)   # pretrained=True 가능
resnet.eval()

dummy_img = torch.randn(1, 3, 224, 224)

In [2]:
# 2. ONNX 변환 + 시간 측정
start = time.time()  # 변환 시작 시간 기록
torch.onnx.export(  # PyTorch 모델을 ONNX 형식으로 변환
    resnet,  # 변환할 ResNet18 모델
    dummy_img,  # 모델 추적(tracing)에 사용할 더미 입력 이미지
    "resnet18.onnx",  # 저장될 ONNX 파일 이름
    input_names=["input"],  # ONNX 모델의 입력 텐서 이름 지정
    output_names=["output"],  # ONNX 모델의 출력 텐서 이름 지정
    opset_version=18,  # 사용할 ONNX opset 버전 지정
    do_constant_folding=True,  # 상수 연산을 미리 계산하여 그래프 최적화
    dynamic_axes={  # 동적 배치 크기 설정
        # 동적 배치(Dynamic Batch)란?
        # 모델을 변환할 때 입력 텐서의 크기를 고정하지 않고,
        # 추론 시점에 배치 크기(batch size)를 자유롭게 변경할 수 있도록 설정하는 것입니다.

        # 예를 들어:
        # - (1, 3, 224, 224) → 이미지 1장
        # - (8, 3, 224, 224) → 이미지 8장
        # - (32, 3, 224, 224) → 이미지 32장

        # 위처럼 첫 번째 차원(0번 차원)을 가변적으로 사용할 수 있게 됩니다.

        # 만약 dynamic_axes를 지정하지 않으면,
        # ONNX 모델은 더미 입력(dummy_img)의 배치 크기(여기서는 1)로 고정됩니다.
        # 즉, (1, 3, 224, 224)만 입력 가능하게 됩니다.
        "input": {0: "batch"},  # 입력 텐서의 0번 차원을 batch로 지정
        "output": {0: "batch"}  # 출력 텐서의 0번 차원을 batch로 지정
    }
)

end = time.time()  # 변환 종료 시간 기록
print(f"[ResNet18] ONNX 변환 완료: {end - start:.3f}초")  # ONNX 변환에 걸린 시간 출력

/tmp/ipykernel_77592/4108774332.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(  # PyTorch 모델을 ONNX 형식으로 변환
W0226 14:13:22.771000 77592 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0226 14:13:22.772000 77592 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0226 14:13:22.773000 77592 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' fro

[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/home/minsu/miniconda3/envs/py310/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 41 of general pattern rewrite rules.
[ResNet18] ONNX 변환 완료: 2.400초


In [3]:
# 3. 이미지 로드 및 전처리
img_path = "image/dog.jpg"   # 실습자가 준비한 이미지 파일

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


img = Image.open(img_path).convert("RGB")
# (1, 3, 224, 224) 형태로 변환
input_tensor = preprocess(img).unsqueeze(0)

In [4]:
# 4. PyTorch 추론 시간 측정
with torch.no_grad():
    start = time.time()
    output_torch = resnet(input_tensor)
    end = time.time()

print(f"[PyTorch] 추론 시간: {end - start:.6f}초")
pred_idx = np.argmax(output_torch, axis=1)[0]
print("예측 클래스 인덱스:", pred_idx)

[PyTorch] 추론 시간: 0.017351초
예측 클래스 인덱스: tensor(259)


In [5]:
# 5. ONNX Runtime 추론 시간 측정 (CPU)
ort_session = ort.InferenceSession(  # ONNX Runtime 세션 생성
    "resnet18.onnx",  # 로드할 ONNX 모델 파일
    providers=["CPUExecutionProvider"]  # CPU를 사용한 추론 설정
)
# # ONNX 입력 형식 준비
# ort_inputs = {"input": input_tensor.numpy().astype("float32")}  # (미사용) 기존 입력 텐서 예시
x = preprocess(img)           # 이미지 전처리 수행 (Resize + ToTensor + Normalize)
x = x.unsqueeze(0)            # 배치 차원 추가 → (1, 3, 224, 224)
x_np = x.cpu().numpy().astype("float32")  # ONNX Runtime 입력을 위해 numpy float32로 변환

ort_inputs = {"input": x_np}  # ONNX 모델 입력 딕셔너리 구성

start = time.time()           # 추론 시작 시간 기록
output_onnx = ort_session.run(None, ort_inputs)  # ONNX Runtime으로 추론 실행
end = time.time()             # 추론 종료 시간 기록

print(f"[ONNX Runtime] 추론 시간: {end - start:.6f}초")  # ONNX Runtime 추론 소요 시간 출력

pred_idx = np.argmax(output_onnx[0], axis=1)[0]
print("예측 클래스 인덱스:", pred_idx)

[ONNX Runtime] 추론 시간: 0.008165초
예측 클래스 인덱스: 259


In [6]:
# 5. ONNX Runtime 추론 시간 측정 (GPU)
ort_session = ort.InferenceSession(  # ONNX Runtime 추론 세션 생성
    "resnet18.onnx",  # 로드할 ONNX 모델 파일 경로
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]  # CUDA 우선, 실패 시 CPU 사용
)
# # ONNX 입력 형식 준비
# ort_inputs = {"input": input_tensor.numpy().astype("float32")}  # (미사용) 기존 입력 텐서 예시
x = preprocess(img)           # 이미지 전처리 수행 (Resize + ToTensor + Normalize)
x = x.unsqueeze(0)            # 배치 차원 추가 → (1, 3, 224, 224)
x_np = x.cpu().numpy().astype("float32")  # GPU/CPU 공용 입력을 위해 numpy float32로 변환

ort_inputs = {"input": x_np}  # ONNX Runtime 입력 딕셔너리 구성

start = time.time()           # 추론 시작 시간 기록
output_onnx = ort_session.run(None, ort_inputs)  # ONNX Runtime으로 추론 실행
end = time.time()             # 추론 종료 시간 기록

print(f"[ONNX Runtime] 추론 시간: {end - start:.6f}초")  # ONNX Runtime 추론 소요 시간 출력
pred_idx = np.argmax(output_onnx[0], axis=1)[0]
print("예측 클래스 인덱스:", pred_idx)

[ONNX Runtime] 추론 시간: 0.229339초
예측 클래스 인덱스: 259


ResNet18 모델 기준으로 **PyTorch vs ONNX Runtime** 추론 성능을 비교했습니다.

결과를 해석할 때는 아래를 함께 봐야 합니다.
- 단일 샘플(batch=1)에서는 I/O와 런타임 오버헤드 비중이 커져 의도와 다른 순위가 나올 수 있습니다.
- GPU 비교는 `CUDAExecutionProvider`가 실제로 활성화된 경우에만 유효합니다.

### 2.6 실습 : ONNX 모델을 TensorRT로 변환

앞 단계에서 우리는 PyTorch → ONNX 변환과 ONNX Runtime 기반 추론 속도 비교까지 진행했습니다.  

이제 ONNX 모델을 **TensorRT 엔진(.plan)** 으로 변환하여 GPU에 최적화된 추론 속도를 확인하는 실습을 진행합니다.

TensorRT는 ONNX Runtime보다 **더 깊은 수준의 최적화(FP16, INT8, 커널 튜닝)** 를 수행하므로  
CNN 모델(특히 ResNet18)에서는 속도 향상이 매우 뚜렷하게 나타납니다.

#### 실습 목표

- 이미 변환한 `resnet18.onnx` 파일을 **TensorRT 엔진(.plan)** 으로 변환  
- TensorRT 런타임으로 추론 속도 측정  
- ONNX Runtime과 비교하여 성능 차이 분석

#### TensorRT 변환이 ONNX보다 빠른 경우가 많은 이유 (요약)

ONNX가 "호환성 중심 포맷"이라면 TensorRT는 **NVIDIA GPU 최적화 중심 엔진**입니다.

TensorRT는 다음을 수행합니다.
- 계산 그래프 재정렬/불필요 연산 제거
- FP16/INT8 혼합 정밀도 최적화
- GPU 아키텍처별 커널 선택(autotuning)
- 레이어 병합(layer fusion)
- 메모리 레이아웃 및 배치 처리 최적화

다만 속도 향상 폭은 모델 구조, 배치 크기, GPU 세대, 엔진 빌드 옵션에 따라 크게 달라집니다.


#### ✔ 코드 예시: ONNX → TensorRT 변환 및 추론

아래 실습은 2단계로 진행합니다.
1. `trtexec` CLI로 ONNX를 TensorRT 엔진으로 변환
2. Python에서 `.plan` 엔진을 로드해 추론 시간 측정

CLI는 환경 이슈를 줄여 엔진 생성 단계가 비교적 안정적이고,
Python API 코드는 추론 파이프라인(메모리 할당/복사/실행)을 학습하기 좋습니다.


#### 1) ONNX → TensorRT 엔진 생성 (CLI)

1. TensorRT 설치 (운영체제에 맞게)
##### 방법 A: 브라우저로 직접 다운로드 (권장)
아래 링크에서 파일 다운  
https://developer.nvidia.com/tensorrt/download/10x   

강의 자료 작성 시점에는 TensorRT 10.13.3 GA for Linux x86_64 and CUDA 12.0 to 12.9 TAR Package 를 설치하였음.

tensorrt_install.sh 파일이 있는곳에 다운받은 파일을 위치한 후 아래 명령어 실행 
```bash
bash tensorrt_install.sh TensorRT-10.13.3.9.Linux.x86_64-gnu.cuda-12.9.tar.gz
```
- 다운로드 후 실습 디렉토리로 이동

##### 방법 B: `wget` 사용 (가능한 경우)

```bash
wget https://developer.nvidia.com/downloads/compute/machine-learning/tensorrt/10.13.3/tars/TensorRT-10.13.3.9.Linux.x86_64-gnu.cuda-12.9.tar.gz
```

> 다운로드가 HTML 몇 KB로 끝나면 로그인 문제이므로  
> 방법 A 사용 권장


#### TensorRT 압축 해제
터미널에서 아래 명령어 실행:
```bash
tar -xvf TensorRT-10.13.3.9.Linux.x86_64-gnu.cuda-12.9.tar.gz -C /usr/local
```

#### `trtexec` 위치 확인

```bash
ls /usr/local/TensorRT-10.13.3.9/targets/x86_64-linux-gnu/bin
```

정상이라면 다음이 보입니다.

```text
trtexec
```

#### 2) ONNX → TensorRT 엔진 변환

```bash
trtexec \
  --onnx=resnet18.onnx \
  --saveEngine=resnet18_fp16.plan \
  --fp16 \
  --memPoolSize=workspace:4096 \
  --verbose
```

옵션 설명:
- `--fp16` : FP16 혼합정밀도 사용 → 속도 향상  
- `--memPoolSize=workspace:4096` : TensorRT 최적화에 사용할 GPU 메모리 (4GB)
- `--saveEngine` : 변환된 TensorRT 엔진 파일 저장
- `--verbose` : 레이어별 최적화 로그 출력 (학습용)  

성공 시 `resnet18_fp16.plan` 파일이 생성되며, 로그 마지막에 다음 문구가 출력됩니다.
```text
&&&& PASSED TensorRT.trtexec
```

#### 3) TensorRT 엔진 불러오기 & 추론 시간 측정

In [7]:
import numpy as np
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit # micromamba install -c conda-forge pycuda
import time
from PIL import Image
from torchvision.transforms import v2

##### 1. 이미지 로드

In [8]:
img_path = "image/dog.jpg"

preprocess = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToTensor(),
])

img = Image.open(img_path).convert("RGB")
input_tensor = preprocess(img).unsqueeze(0).numpy()   # (1,3,224,224)

/home/minsu/miniconda3/envs/py310/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


##### 2. TensorRT 엔진 로드


In [9]:
TRT_LOGGER = trt.Logger()  # TensorRT 실행 중 메시지를 출력하기 위한 로그 도구 생성
with open("resnet18_fp16.plan", "rb") as f:  # 미리 만들어둔 TensorRT 모델 파일을 읽기 모드로 열기
    engine = trt.Runtime(TRT_LOGGER).deserialize_cuda_engine(f.read())  # 파일에 저장된 모델을 메모리로 불러오기

context = engine.create_execution_context()  # 모델을 실제로 실행하기 위한 실행 환경 생성

##### 3. 메모리 할당 

아래 코드를 살펴보면, 입력·출력 텐서의 이름을 직접 조회하고, 입력 크기를 명시적으로 설정하며,  
GPU 메모리를 수동으로 할당하는 등 일반적인 딥러닝 프레임워크에서 자동으로 처리되는 작업들을  
사용자가 직접 수행해야 함을 확인할 수 있습니다.  

이는 TensorRT가 최소한의 런타임 오버헤드와 최대 추론 성능을 목표로 설계되었기 때문입니다.  
TensorRT는 내부에서 추론 과정에 필요한 모든 정보를 자동으로 추론하기보다는,  
모델의 입·출력 구조와 메모리 사용을 명확하게 지정받아 고정된 실행 경로를 구성합니다.  

그 결과, 코드 작성 단계에서는 다소 복잡해질 수 있지만,  
추론 시점에서는 불필요한 검사나 메모리 재할당이 제거되어 지연 시간(latency)이 크게 감소하고  
GPU 자원을 보다 효율적으로 활용할 수 있습니다.  

In [10]:
TRT_LOGGER = trt.Logger()  # TensorRT 실행 중 메시지를 출력하기 위한 로그 도구 생성
with open("resnet18_fp16.plan", "rb") as f:  # 미리 변환해둔 TensorRT 모델 파일을 읽기 모드로 열기
    engine = trt.Runtime(TRT_LOGGER).deserialize_cuda_engine(f.read())  # 저장된 모델을 GPU에서 사용할 수 있도록 불러오기

context = engine.create_execution_context()  # 추론을 수행할 실행 환경 생성

input_name = engine.get_tensor_name(0)  # 모델이 받는 입력 데이터의 이름 가져오기
output_name = engine.get_tensor_name(1)  # 모델이 내보내는 출력 데이터의 이름 가져오기

context.set_input_shape(input_name, input_tensor.shape)  # 입력 데이터 크기를 모델에 알려줌
output_shape = context.get_tensor_shape(output_name)  # 출력 데이터의 크기 확인

d_input = cuda.mem_alloc(input_tensor.nbytes)  # 입력 데이터를 GPU에 올리기 위한 메모리 공간 확보
d_output = cuda.mem_alloc(int(np.prod(output_shape)) * 4)  # 출력 데이터를 GPU에서 저장할 공간 확보(float32 기준)

stream = cuda.Stream()  # GPU 작업을 빠르게 처리하기 위한 비동기 처리 통로 생성

[02/26/2026-14:13:25] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.


##### 4. TensorRT 추론 시간 측정

In [11]:
context.set_tensor_address(input_name, int(d_input))   # TensorRT 컨텍스트에 입력 텐서가 위치한 GPU 메모리 주소 등록
context.set_tensor_address(output_name, int(d_output)) # TensorRT 컨텍스트에 출력 텐서가 위치할 GPU 메모리 주소 등록
context.execute_async_v3(stream_handle=stream.handle)  # 지정한 CUDA Stream에서 비동기 방식으로 추론 실행

True

In [12]:
# 입력 복사 (Host → Device)
cuda.memcpy_htod_async(d_input, input_tensor, stream)  # CPU 메모리의 입력 데이터를 GPU 메모리로 비동기 복사

start = time.time()                                    # 추론 시작 시간 기록

# === TensorRT 10.x 실행 (최소 변경) ===
context.set_tensor_address(input_name, int(d_input))   # 입력 GPU 메모리 주소를 다시 컨텍스트에 설정
context.set_tensor_address(output_name, int(d_output)) # 출력 GPU 메모리 주소를 다시 컨텍스트에 설정

context.execute_async_v3(stream_handle=stream.handle)  # CUDA Stream을 사용해 비동기 추론 실행

stream.synchronize()                                   # 모든 GPU 작업이 끝날 때까지 대기
end = time.time()                                      # 추론 종료 시간 기록

# 출력 복사 (Device → Host)
output = np.empty(output_shape, dtype=np.float32)      # 출력 결과를 받을 CPU 메모리 공간 생성
cuda.memcpy_dtoh_async(output, d_output, stream)       # GPU 메모리의 출력 데이터를 CPU 메모리로 비동기 복사
stream.synchronize()                                   # 복사 작업이 완료될 때까지 대기

print(f"[TensorRT] 추론 시간: {end - start:.6f}초")    # TensorRT 추론에 소요된 시간 출력
pred_idx = np.argmax(output, axis=1)[0]
print("예측 클래스 인덱스:", pred_idx)

[TensorRT] 추론 시간: 0.000358초
예측 클래스 인덱스: 259


#### ✔ 요약

- ONNX는 호환성이 강점이고, TensorRT는 NVIDIA GPU 최적화가 강점입니다.
- TensorRT 성능 평가는 단일 1회 시간이 아니라 워밍업 + 반복 평균으로 보는 것이 안전합니다.
- batch=1에서는 I/O 오버헤드 비중이 커서 기대보다 이득이 작게 보일 수 있습니다.
- 실제 배포 환경에서는 batch, 동시성, 입력 크기, 정밀도(FP16/INT8)를 함께 조정해 최적점을 찾습니다.


## 3. 양자화 (Quantization)

양자화(Quantization)는 딥러닝 모델이 사용하는 **숫자의 표현 범위를 줄여**  
모델 크기를 작게 하고 추론 속도를 높이는 기술입니다.

일반적으로 모델은 **FP32(32비트 부동소수점)** 을 사용하지만,  
양자화를 통해 **FP16(16비트) / INT8(8비트)** 와 같이 더 작은 비트로 표현할 수 있습니다.

### 3.1 양자화의 개념과 장점

딥러닝 모델에서 숫자를 표현할 때 “비트를 줄인다”는 것은  
더 적은 메모리로 가중치를 저장하고, 더 간단한 연산으로 계산한다는 의미입니다.

**비트 수를 줄이면 생기는 효과는 크게 세 가지입니다:**

#### ✔ 1) 모델 크기 감소
- FP32 → INT8 로 변환하면 **이론적으로 4배** 모델 크기가 줄어듦  
- 모바일·엣지·서버 비용 절감

#### ✔ 2) 추론 속도 향상
- 숫자가 작아질수록 CPU/GPU에서 **한 번에 더 많은 연산 병렬 처리 가능**  
- 특히 TensorRT, ONNX Runtime, TFLite 등에서 가속 효과가 큼

#### ✔ 3) 메모리 대역폭 절약
- 작은 비트는 메모리 읽기/쓰기 양이 줄어 전체 처리 속도를 높임  
- GPU 메모리 한계를 넘지 않고 더 큰 모델도 실행 가능

#### Post-Training Quantization (PTQ) vs Quantization-Aware Training (QAT)

양자화는 크게 **훈련 후 적용(PTQ)** 과  
**훈련 과정에서 양자화를 고려(QAT)** 하는 방식으로 나눌 수 있습니다.

### 3.2 PTQ (Post-Training Quantization)

이미 학습이 끝난 FP32 모델을 **재훈련 없이** INT8/FP16으로 변환하는 방식입니다.  
이때 값의 범위를 줄이기 위해 **정적(static) 또는 동적(dynamic) 스케일링**을 사용합니다.

#### 3.2.1 Post-Training Static Quantization (PTQ Static)
정적 방식에서는 모델의 각 레이어 출력값을 **샘플 데이터로 미리 측정**하여  
출력의 최소값과 최대값을 찾아내고, 그 범위를 기준으로 고정된 scale을 계산합니다.  
이 과정을 **캘리브레이션(calibration)** 이라고 합니다.

#### 건물 높이 비유
도시의 건물 높이를 INT8로 표현한다고 가정합니다.

- 가장 낮은 건물: **10m**  
- 가장 높은 건물: **828m(부르즈 할리파)**  
- 높이 전체 범위: **818m**

이 큰 범위를 INT8의 256칸(-128~127)에 맞추기 위해  
다음과 같은 scale을 계산합니다.

<img src="image/Static_Quantization.png" width="600">

```
scale = 818m / 256 ≈ 3.19m per step
```

이제 모든 건물 높이는 다음처럼 정수로 변환됩니다.

```
int8_height = round((real_height - 10m) / scale)
```


이때 실제 높이를 3m 단위로만 표현하게 되므로  
“세밀한 높이 차이”는 일부 손실될 수 있습니다.  
모델에서도 이와 같은 방식으로 activation을 고정된 scale을 이용해 INT8으로 변환합니다.

#### Post-Training Static Quantization (PTQ Static) 추론 흐름

1️ **오프라인(모델 준비 단계)**  
- 샘플 데이터로 각 레이어 activation의 min/max 측정 (캘리브레이션)  
- 레이어별 고정 scale 계산  
- 가중치를 INT8로 변환  

2️ **실제 추론 단계**  
- 입력 이미지 → 고정된 input scale로 INT8 변환  
- INT8 입력 × INT8 가중치 → INT32로 누적 연산  
- 레이어 연산 종료 후 scale을 적용해 다시 INT8로 재양자화  
- 다음 레이어로 전달 (반복)  
- 최종 출력 생성  

👉 즉, **가중치와 입력은 INT8**,  
👉 **연산 중 누적은 INT32**,  
👉 **레이어마다 scale을 이용해 다시 INT8로 압축**하는 구조입니다.

#### 3.2.2 동적 양자화 Post-Training Dynamic Quantization (PTQ Dynamic)

동적 양자화는 사전 캘리브레이션 없이,  
**행렬곱(MatMul / Linear) 연산이 실행되는 순간에만 activation을 INT8로 변환**하는 방식입니다.

중요한 점은:

activation이 계속 INT8로 유지되는 것이 아니라  
연산 직전에만 INT8로 바뀌었다가, 연산 직후 다시 FP32로 복원된다는 것입니다.

##### 동적 양자화의(Dynamic Quantization) 실행 흐름

① 입력 데이터 준비  
- 전처리된 입력은 **FP32 텐서 상태**

② 첫 번째 Linear/MatMul 레이어 실행 직전  
- 현재 값은 "입력 데이터(FP32)"
- 이 순간 해당 텐서의 **min/max 계산**
- 즉석에서 scale 계산
- 입력 데이터를 **INT8로 변환**

③ Linear 연산 수행  
- INT8 입력 × INT8 가중치  
- 결과는 INT32로 누적(accumulate)

④ Linear 연산 직후  
- INT32 결과를 scale을 이용해 **FP32로 복원**
- 이 값이 이제 "activation"

⑤ 다음 레이어 진행  
- ReLU/GELU 등은 FP32로 수행
- 다음 Linear 레이어 직전  
  → 현재 activation(FP32)의 min/max 계산  
  → 다시 INT8로 변환  
  → 연산 수행

#### Static vs Dynamic 비교 요약

| 구분 | Static Quantization | Dynamic Quantization |
|------|----------------------|------------------------|
| 사전 캘리브레이션 | 필요 (activation 통계 수집) | 필요 없음 |
| scale 계산 시점 | 캘리브레이션 단계에서 고정 | 실행 시점에 activation 기준 계산 |
| weight | INT8 (사전 양자화) | INT8 (사전 양자화) |
| activation | INT8로 양자화 후 연산 (INT8 유지 가능) | 연산 직전에 FP32→INT8, 연산 후 FP32 복원 |
| scale | weight/activation scale 모두 고정 | weight scale 고정 / activation scale 동적 |
| 장점 | Full INT8 파이프라인 가능, CNN에 효율적 | 구현 단순, Transformer/Linear 모델에 적합 |
| 단점 | 캘리브레이션 품질 의존 | 매 레이어 min/max 계산 오버헤드 존재 |

#### ✔ 핵심 요약
- **Static Quantization**은 “사용될 activation 범위를 미리 조사하여 고정 scale을 설정하는 방식”입니다.  
- **Dynamic Quantization**은 “실행 시 매 matmul 연산 직전에 필요한 activation 범위만 보고 scale을 계산하는 방식”입니다.

### 3.3 QAT (Quantization-Aware Training)

QAT(Quantization-Aware Training)는 모델을 학습할 때부터 **INT8 환경을 가짜로 시뮬레이션**하면서 훈련시키는 방식입니다.

- 사후 양자화(PTQ, Static/Dynamic)는 이미 학습이 끝난 FP32 모델을 나중에 INT8로 변환합니다.
- QAT는 학습 중간에 “INT8처럼 보이게 만드는 연산(fake quantization)”을 넣어서  
  모델이 **양자화로 인한 오차를 스스로 보정하도록** 학습합니다.

훈련이 끝난 후에는 fake 연산을 제거하고  
실제로 weight·activation을 INT8 형식으로 내보냅니다(export).

#### 건물 높이 비유로 보는 QAT

정수(INT8) 양자화를 “건물 높이를 제한된 눈금으로 표현하는 과정”으로 생각해 봅니다.

- FP32: 0.000001m 단위까지 측정 가능한 고급 장비  
- INT8: 256칸으로만 높이를 표시하는 거친 눈금자

사후 양자화(Static/Dynamic)는 이미 지어진 건물(완성된 FP32 모델)을  
나중에 256칸 눈금으로 억지로 표시하는 것입니다.  
이때 건물 높이 정보가 뭉개져서(정보 손실) 모델 성능이 떨어질 수 있습니다.

QAT는 여기서 접근을 바꿉니다.

> 건물이 처음 지어지는 설계 단계부터 “나중에 256칸 눈금자로 기록될 것”을 알고  
>  그 제약을 고려해서 구조를 최적화하는 것과 같습니다.

즉, 모델이 학습을 진행하는 동안 “INT8로 표현되면 값이 어떻게 잘리고, 어떤 왜곡이 생기는지”를  
직접 겪으면서 그에 맞게 weight를 조정합니다.

#### Fake Quantization의 동작 원리

QAT의 핵심은 **Fake Quantization(가짜 양자화)** 연산입니다.

실제 INT8로 바꾸지는 않지만, 아래 과정을 통해 **INT8처럼 보이게 만든 뒤 다시 FP32로 되돌립니다.**

1. 현재 FP32 텐서의 값 범위(min, max)를 사용해 scale을 계산합니다.
2. 이 scale로 FP32 값을 INT8 범위로 양자화합니다.
   - 예:  
     `q = round(x_fp32 / scale)`  
     `q`는 실제로는 FP32 타입이지만 값은 INT8처럼 제한된 정수 형태가 됩니다.
3. 다시 FP32 범위로 되돌립니다(dequantize).
   - `x_fake = q * scale`
4. 이후 연산(다음 레이어, 손실 계산 등)은 `x_fake`(이미 왜곡된 값)를 사용하여 진행합니다.

즉, forward에서는:

> FP32 값 → INT8처럼 잘라냄 → 다시 FP32로 복원  
> 이 상태를 입력으로 학습을 계속 진행합니다.

이 과정을 weight 주변, activation 주변에 적절히 삽입합니다.


#### 훈련 흐름(수식 관점 정리)

일반적인 FP32 훈련과 비교하면 다음과 같습니다.

#### 1) 일반 FP32 훈련

- Forward:  `y = f(x; W_fp32)`  
- Backward:  손실 `L(y)`에 대해 `W_fp32`를 업데이트

#### 2) QAT 훈련

QAT에서는 forward에 Fake Quantization이 끼어 들어갑니다.

1. 가중치 fake quantization  
   - `W_q = Q(W_fp32)`   (INT8처럼 양자화한 뒤 FP32로 표현)
   - 실질적으로는 `Q` 연산이 `round + clipping`을 수행
   - 예시 : 0.4429 → 44 → 0.44
   
2. activation(이전 레이어의 출력 또는 토큰 임베딩) fake quantization  
   - `x_q = Q(x_fp32)` , x_fp32 : 이전 레이어의 출력  

3. 양자화된 값으로 연산  
   - `y = f(x_q; W_q)`  
   - (실제 연산은 FP32로 수행되지만, 값은 INT8처럼 왜곡된 상태)

4. backward 시에는  
    - 양자화는 반올림 때문에 수학적으로 미분이 어렵습니다.  
    - 그래서 학습할 때는  
    backward에서는 양자화를 잠시 무시하고 gradient를 그대로 흘려보냅니다.  
    - 이 편법적인 처리 방식을 Straight-Through Estimator(STE)라고 합니다.

이렇게 하면:

- forward에서는 INT8 세계의 왜곡(라운딩, 클리핑)을 경험하고  
- backward에서는 그 왜곡을 보정하는 방향으로 weight가 업데이트됩니다.


#### QAT와 PTQ(Static/Dynamic)의 차이

정리하면 다음과 같은 큰 차이가 있습니다.

1. PTQ(Static/Dynamic)는  
   - 학습이 끝난 FP32 모델을 나중에 INT8로 바꿉니다.  
   - 모델이 양자화를 “모른 채” 학습이 끝난 상태입니다.  
   - 양자화 후 정확도가 떨어져도, 이를 다시 보정할 수단이 제한적입니다.

2. QAT는  
   - 학습하는 동안부터 INT8 환경을 흉내 냅니다.  
   - weight와 activation이 “항상 양자화된 값처럼 보이는 상태”에서 학습합니다.  
   - 최종적으로 INT8로 내보내더라도 이미 그 환경에 최적화된 파라미터를 가지고 있습니다.


#### 건물 높이 비유로 다시 한 번 비교

- 사후 양자화(PTQ):  
  이미 완성된 건물을 나중에 256칸 눈금자로만 기록하는 상황입니다.  
  세밀한 디자인(파라펫, 옥탑 구조 등)이 많이 손실될 수 있습니다.

- QAT:  
  처음 설계, 시공 단계부터 “완공 후에는 256칸 눈금자로 기록될 것”이라는 조건을 알고  
  그 조건을 만족하도록 설계와 구조를 조정합니다.  
  그래서 나중에 실제로 256칸으로 기록해도 의도했던 형태가 잘 유지됩니다.

#### 장점

- INT8 모델에서 **FP32에 매우 근접한 정확도**를 얻을 수 있습니다.
- attention, GELU, LayerNorm, residual connection 등  
  정밀한 연산이 많은 LLM, ViT, 대형 CNN에 특히 유리합니다.
- Static/Dynamic PTQ에서 큰 정확도 손실이 나는 경우  
  QAT를 적용하면 상당 부분 회복되거나 거의 동일 수준까지 끌어올릴 수 있습니다.

#### 단점

- Fake Quantization 연산이 추가되므로 **훈련 시간이 더 길어지고, 계산량이 증가**합니다.
- 기존 FP32-only 학습 파이프라인에 QAT용 모듈(fq 모듈, fake quant node 등)을 삽입하는 작업이 필요합니다.
- 이미 학습이 끝난 모델에 “추가 미세 튜닝”을 QAT로 수행하는 경우에도 일정량의 재훈련 시간이 필요합니다.

#### ✔ 두 방식 비교 요약

| 방법 | 장점 | 단점 | 사용 목적 |
|------|------|------|-----------|
| **PTQ** | 빠름, 쉬움, 재훈련 필요 없음 | 정확도 손실 가능 | 빠른 배포, CNN/NLP |
| **QAT** | 정확도 손실 최소화 | 훈련 비용 증가 | 정확한 모델 유지 필요 |

### 3.3 PyTorch ResNet18 양자화 (PTQ – 동적/정적) 실습

이 실습의 핵심은 "INT8이 항상 빠르다"를 증명하는 것이 아니라,
**어떤 조건에서 빨라지고 어떤 조건에서 느려지는지 측정으로 확인**하는 것입니다.

In [13]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import time

# 1. 원본 FP32 PyTorch 모델 준비
# ResNet18 모델 불러오기 (pretrained=True 가능)
model_fp32 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model_fp32.eval()

print("[1] FP32 모델 준비 완료")

[1] FP32 모델 준비 완료


In [14]:
# 2. 테스트 이미지 로딩 + 전처리
img_path = "image/dog.jpg"

# ResNet 입력 형태에 맞춘 전처리
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

img = Image.open(img_path).convert("RGB")
input_tensor = preprocess(img).unsqueeze(0)   # (1, 3, 224, 224)

print(f"[2] 입력 텐서 shape: {input_tensor.shape}")

[2] 입력 텐서 shape: torch.Size([1, 3, 224, 224])


In [15]:
# 3. FP32 모델 추론 시간 측정
with torch.no_grad():
    start = time.time()
    output_fp32 = model_fp32(input_tensor)
    end = time.time()

print(f"[3] FP32 모델 추론 시간: {end - start:.6f}초")

[3] FP32 모델 추론 시간: 0.013433초


In [16]:
# 4. 동적 양자화 (Dynamic Quantization)
# ResNet은 Conv가 많아서 동적 양자화 효과가 제한적.
#  → 주로 Linear 레이어에만 적용됨.

model_dynamic = torch.quantization.quantize_dynamic(
    model_fp32,                    # 양자화 대상 모델
    {torch.nn.Linear},             # Linear 레이어만 INT8 양자화
    dtype=torch.qint8              # INT8 양자화 수행
)

print("[4] 동적 양자화 모델 생성 완료")

[4] 동적 양자화 모델 생성 완료


/tmp/ipykernel_77592/1346016707.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_dynamic = torch.quantization.quantize_dynamic(


In [17]:
# 추론 시간 비교
with torch.no_grad():
    start = time.time()
    output_dynamic = model_dynamic(input_tensor)
    end = time.time()

print(f"[동적 PTQ] 추론 시간: {end - start:.6f}초")

[동적 PTQ] 추론 시간: 0.013289초


In [18]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType  # ONNX Runtime의 정적(INT8) 양자화 관련 모듈
import onnx                               # ONNX 모델 로딩 및 처리용 라이브러리
from PIL import Image                    # 이미지 로딩을 위한 PIL
import numpy as np                       # 수치 연산 및 배열 처리를 위한 NumPy
import torchvision.transforms as transforms  # 이미지 전처리를 위한 torchvision 변환 도구
import os                                # 파일 및 디렉터리 처리용 모듈

# 1. Calibration 데이터 리더 정의
class ResNetDataReader(CalibrationDataReader):  # 정적 양자화를 위한 Calibration 데이터 리더 클래스
    def __init__(self, image_folder):
        self.preprocess = transforms.Compose([  # 입력 이미지 전처리 파이프라인 정의
            transforms.Resize((224, 224)),      # ResNet 입력 크기에 맞게 이미지 리사이즈
            transforms.ToTensor(),               # 이미지를 Tensor로 변환
        ])
        self.image_paths = [                     # 지정한 폴더에서 이미지 파일 경로 수집
            os.path.join(image_folder, f)
            for f in os.listdir(image_folder)
            if f.lower().endswith((".jpg", ".png"))
        ]
        self.datas = []                          # 전처리된 입력 데이터를 저장할 리스트
        for img_path in self.image_paths:        # 각 이미지에 대해
            img = Image.open(img_path).convert("RGB")  # 이미지를 RGB 형식으로 로드
            x = self.preprocess(img).unsqueeze(0).numpy().astype(np.float32)  # 전처리 + 배치 차원 추가 + numpy 변환
            self.datas.append({"input": x})      # ONNX 입력 이름에 맞춰 딕셔너리 형태로 저장
        self.enum_data = iter(self.datas)         # 데이터를 순차적으로 반환하기 위한 iterator 생성

    def get_next(self):
        return next(self.enum_data, None)         # 다음 calibration 데이터 반환, 없으면 None 반환


# 2. Static Quantization 실행

fp32_model = "resnet18.onnx"                      # FP32(원본) ONNX 모델 경로
int8_model = "resnet18_int8_static.onnx"          # INT8로 양자화된 ONNX 모델 저장 경로

# calibration 데이터 (일반적으로 3~10장 정도의 이미지면 충분)
reader = ResNetDataReader("image")                # calibration에 사용할 이미지 폴더 지정

quantize_static(
    model_input=fp32_model,                       # 입력 FP32 ONNX 모델
    model_output=int8_model,                      # 출력 INT8 ONNX 모델
    calibration_data_reader=reader,               # calibration 데이터 리더
    weight_type=QuantType.QInt8,                  # 가중치를 INT8 형식으로 양자화
)

print("Static INT8 양자화 완료:", int8_model)     # 양자화 완료 메시지 출력

Static INT8 양자화 완료: resnet18_int8_static.onnx


In [19]:
import onnxruntime as ort
import numpy as np
import time

# 이미지 하나 로드
img = Image.open("image/dog.jpg").convert("RGB")          # 테스트용 이미지 로드 후 RGB로 변환
x = transforms.Resize((224,224))(img)                    # ResNet 입력 크기에 맞게 이미지 리사이즈
x = transforms.ToTensor()(x).unsqueeze(0).numpy().astype(np.float32)  # Tensor 변환 + 배치 차원 추가 + numpy(float32) 변환

In [20]:
sess_fp32 = ort.InferenceSession("resnet18.onnx")         # FP32 ONNX 모델을 위한 ONNX Runtime 세션 생성
sess_int8 = ort.InferenceSession("resnet18_int8_static.onnx")  # INT8 Static Quantization ONNX 모델 세션 생성

# FP32 추론
start = time.time()                                      # FP32 추론 시작 시간 기록
out_fp32 = sess_fp32.run(None, {"input": x})             # FP32 모델로 추론 실행
t_fp32 = time.time() - start                             # FP32 추론 소요 시간 계산

# INT8 추론
start = time.time()                                      # INT8 추론 시작 시간 기록
out_int8 = sess_int8.run(None, {"input": x})             # INT8 모델로 추론 실행
t_int8 = time.time() - start                             # INT8 추론 소요 시간 계산

print("FP32:", t_fp32)                                   # FP32 모델 추론 시간 출력
print("INT8:", t_int8)                                   # INT8 모델 추론 시간 출력

pred_fp32 = np.argmax(out_fp32[0], axis=1)               # FP32 모델 출력(logits)에서 가장 높은 값을 갖는 클래스 인덱스 추출
pred_int8 = np.argmax(out_int8[0], axis=1)               # INT8 모델 출력(logits)에서 가장 높은 값을 갖는 클래스 인덱스 추출

print("FP32 prediction:", pred_fp32)                     # FP32 모델의 최종 예측 클래스 출력
print("INT8 prediction:", pred_int8)                     # INT8 모델의 최종 예측 클래스 출력

FP32: 0.010653018951416016
INT8: 0.015258073806762695
FP32 prediction: [259]
INT8 prediction: [259]


In [21]:
sess_fp32 = ort.InferenceSession("resnet18.onnx",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
) # FP32 ONNX 모델을 위한 ONNX Runtime 세션 생성

sess_int8 = ort.InferenceSession("resnet18_int8_static.onnx",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
) # INT8 Static Quantization ONNX 모델 세션 생성

# FP32 추론
start = time.time()                                      # FP32 추론 시작 시간 기록
out_fp32 = sess_fp32.run(None, {"input": x})             # FP32 모델로 추론 실행
t_fp32 = time.time() - start                             # FP32 추론 소요 시간 계산

# INT8 추론
start = time.time()                                      # INT8 추론 시작 시간 기록
out_int8 = sess_int8.run(None, {"input": x})             # INT8 모델로 추론 실행
t_int8 = time.time() - start                             # INT8 추론 소요 시간 계산

print("FP32:", t_fp32)                                   # FP32 모델 추론 시간 출력
print("INT8:", t_int8)                                   # INT8 모델 추론 시간 출력

pred_fp32 = np.argmax(out_fp32[0], axis=1)               # FP32 모델 출력(logits)에서 가장 높은 값을 갖는 클래스 인덱스 추출
pred_int8 = np.argmax(out_int8[0], axis=1)               # INT8 모델 출력(logits)에서 가장 높은 값을 갖는 클래스 인덱스 추출

print("FP32 prediction:", pred_fp32)                     # FP32 모델의 최종 예측 클래스 출력
print("INT8 prediction:", pred_int8)                     # INT8 모델의 최종 예측 클래스 출력

2026-02-26 14:13:26.166085501 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 21 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.


FP32: 0.017992734909057617
INT8: 0.01800537109375
FP32 prediction: [259]
INT8 prediction: [259]


#### 왜 어떤 환경에서는 ResNet18 INT8이 FP32보다 느려질까?

이 질문의 핵심은 "INT8이 항상 빠르다"가 아니라,
**실행 런타임과 하드웨어 조건이 INT8 최적화를 얼마나 지원하느냐**입니다.

또한 실험 결과는 매번 조금씩 달라질 수 있습니다.
- 배치 크기(batch)
- provider(CPU/CUDA/TensorRT)
- CPU 명령어 지원(AVX512 VNNI 등)
- GPU/드라이버/라이브러리 버전
- 측정 방식(1회 vs 반복 평균, I/O 포함 여부)

따라서 이 실습은 고정된 숫자 암기보다,
**동일 조건에서 신뢰도 있는 비교 절차를 익히는 것**을 목표로 합니다.


##### ✔ 1. CPU 환경

CPU INT8이 빨라지려면 하드웨어와 런타임이 INT8 경로를 잘 지원해야 합니다.
지원이 약한 환경에서는 quant/dequant 오버헤드 때문에 FP32보다 느려질 수 있습니다.

실무 해석 팁:
- 평균(ms)뿐 아니라 표준편차까지 함께 본다.
- 단일 샘플(batch=1) 결과만으로 최종 결론을 내리지 않는다.


##### ✔ 2. GPU 환경(ONNX Runtime CUDA)

CUDA EP를 쓴다고 해서 모든 모델의 INT8이 자동으로 크게 빨라지는 것은 아닙니다.
런타임이 어떤 커널 경로를 선택하는지, fallback이 발생하는지에 따라 결과가 달라집니다.

반드시 확인할 것:
- `ort.get_available_providers()`
- 세션의 `get_providers()` 출력

GPU 비교는 위 두 조건이 만족될 때만 유효합니다.


##### ✔ 3. batch=1과 I/O 오버헤드 영향

batch=1 환경에서는 다음 비용이 상대적으로 크게 보입니다.
- Host↔Device 메모리 복사
- 런타임 준비/커널 런치 비용
- quant/dequant 부가 비용

그래서 모델 연산이 가벼울수록 "순수 연산 이득"보다 "주변 오버헤드"가 커져,
INT8 이득이 작거나 역전될 수 있습니다.


##### ✔ 4. 모델 구조와 실행 엔진 궁합

ResNet18은 Conv 중심 구조라 이론적으로 INT8 친화적이지만,
실제 속도는 "모델 자체"보다 "실행 엔진 최적화 수준"에 크게 좌우됩니다.

즉, 같은 INT8 모델이라도
- ONNX Runtime CPU/CUDA
- TensorRT
환경에 따라 결과가 달라지는 것이 정상입니다.




##### ✔ 5. TensorRT 기반 GPU INT8은 완전히 다른 결과를 보임

동일한 ResNet18 모델이라도 **TensorRT를 사용하여 GPU INT8로 실행할 경우**,  
성능 결과는 완전히 달라집니다.

- FP32 대비 **3~10배 이상의 속도 향상**
- Tensor Core 적극 활용
- 메모리 사용량 감소
- 전체 처리량(throughput) 대폭 증가

이는 **ONNX Runtime과 TensorRT의 엔진 설계 목적 차이**에서 기인합니다.



##### 정리

- 실험 결과가 매번 조금씩 달라지는 것은 자연스러운 현상입니다.
- 단일 샘플/1회 측정은 오해를 만들기 쉬우므로, 워밍업 + 반복 평균으로 비교합니다.
- I/O 오버헤드(메모리 복사, 런타임 준비)를 포함할지 여부를 먼저 정하고 비교해야 합니다.
- GPU 비교는 CUDA provider 활성화 여부를 확인한 뒤 해석해야 합니다.
- INT8의 성능 이득은 "항상"이 아니라 "조건부"이며, 실제 배포 조건에서 재측정이 필수입니다.


### 3.4 양자화 후 성능 평가

양자화를 적용한 후 반드시 모델 성능을 확인해야 합니다.  

- 테스트 데이터셋으로 **정확도(accuracy) 또는 손실(loss)** 확인
- PTQ의 경우 **약간의 성능 손실**이 있을 수 있으므로, 허용 범위 내인지 검증
- QAT는 훈련 단계에서 최적화되므로 성능 저하가 거의 없음

> Tip: 양자화 적용 후 성능이 많이 떨어지면, QAT로 재훈련하거나 일부 레이어만 양자화 적용 가능


## 복습 문제

### 문제 1. 모델 변환
PyTorch 모델을 ONNX로 변환하는 방법을 설명하고, 해당 코드 예시를 작성하시오.

<details> <summary>정답 보기</summary>

PyTorch 모델을 ONNX(Open Neural Network Exchange) 형식으로 변환하면  
다양한 추론 엔진(ONNX Runtime, TensorRT 등)에서 모델을 실행할 수 있습니다.

변환 절차는 다음과 같습니다.

1. 학습된 모델을 로드한다.
2. eval() 모드로 전환한다. (Dropout, BatchNorm 등의 동작을 고정하기 위함)
3. 모델 구조를 추적하기 위한 더미 입력(dummy input)을 생성한다.
4. torch.onnx.export() 함수를 사용해 ONNX 파일로 저장한다.

```python
# 1. 학습된 모델 로드
model = torch.load('model.pth')

# 2. 추론 모드로 전환 (필수)
model.eval()

# 3. 모델 구조를 추적하기 위한 더미 입력 생성
# (batch_size=1, channel=3, height=224, width=224)
dummy_input = torch.randn(1, 3, 224, 224)

# 4. ONNX 형식으로 모델 변환 및 저장
torch.onnx.export(
    model,                 # 변환할 PyTorch 모델
    dummy_input,           # 모델 구조 추적용 입력
    'model.onnx',          # 저장할 ONNX 파일명
    verbose=True           # 변환 과정 출력
)
```

### 문제 2. 양자화
양자화(Post-Training Quantization)와 Quantization-Aware Training(QAT)의 차이를 설명하시오.

<details> <summary>정답 보기</summary>

- **PTQ**: 훈련이 끝난 후 모델을 양자화하는 방법입니다. 기존 모델을 그대로 사용할 수 있습니다.
- **QAT**: 훈련 중 양자화를 고려하여 모델을 최적화하는 방법입니다.

</details>

### 문제 3. 속도 측정 실험 설계
모델 추론 속도 비교 실험에서 **1회 측정만으로 결론을 내리면 왜 위험한지** 설명하고, 
신뢰도 있는 비교를 위해 어떤 절차(예: warm-up, 반복 측정, 평균/표준편차)를 써야 하는지 작성하시오.

<details> <summary>정답 보기</summary>

1회 측정은 초기화 오버헤드, OS 스케줄링, 캐시 상태, GPU 커널 준비 비용 등에 크게 영향을 받아 편차가 큽니다.  
그래서 단일 값만 보고 FP32/INT8 또는 PyTorch/ONNX/TensorRT 우열을 판단하면 오판할 수 있습니다.

권장 절차:
1. 동일 입력/동일 배치 크기로 조건 고정
2. warm-up을 여러 번 수행해 초기화 비용 분리
3. 본 측정은 반복 수행 후 평균(mean)과 표준편차(std) 기록
4. 가능하면 p50/p90/p99 지표까지 함께 확인

예시 코드:
```python
times = []
for _ in range(10):
    run()  # warm-up
for _ in range(100):
    t0 = time.perf_counter()
    run()
    t1 = time.perf_counter()
    times.append((t1 - t0) * 1000)
print(np.mean(times), np.std(times))
```

</details>

### 문제 4. 속도 측정 실험에서 자주 발생하는 예외
다음 상황이 생기면 결과 해석이 왜 왜곡될 수 있는지 설명하시오.

1) ONNX Runtime에서 `CUDAExecutionProvider`를 지정했는데 실제로는 CPU로 fallback 된 경우  
2) batch=1 단일 입력에서 GPU가 CPU보다 느리게 나온 경우  
3) TensorRT 측정 시 H2D/D2H 복사를 제외하고 순수 커널 시간만 측정한 경우  

<details>
<summary>정답 보기</summary>

1) CUDA provider fallback  
- 코드에 GPU 실행을 지정했더라도, 실제 환경에서 CUDA provider가 활성화되지 않으면 CPU로 실행될 수 있습니다.  
- 이 상태에서 측정한 결과를 "GPU 성능"이라고 해석하면 잘못된 결론이 됩니다.  
- 반드시 `ort.get_available_providers()` 등을 통해 실제 실행 장치를 확인해야 합니다.

2) batch=1에서 GPU가 느린 경우  
- GPU는 대량의 연산을 병렬 처리할 때 강점을 보입니다.  
- batch=1처럼 연산량이 작은 경우에는 커널 실행 준비 시간이나 데이터 복사 시간이 상대적으로 크게 작용할 수 있습니다.  
- 따라서 “GPU는 항상 CPU보다 빠르다”는 단정은 옳지 않습니다.

3) TensorRT에서 순수 커널 시간만 측정한 경우  
- H2D (Host → Device, CPU에서 GPU로 데이터 이동)  
- D2H (Device → Host, GPU에서 CPU로 결과 이동)  
시간을 제외하면 실제 서비스 지연시간보다 더 빠르게 보입니다.  
- 다른 프레임워크는 전체 처리 시간(E2E, End-to-End)을 측정했는데 TensorRT만 순수 연산 시간만 측정했다면 공정한 비교가 아닙니다.

정리:  
속도 측정 시에는 **어디까지 시간을 포함했는지(E2E인지, 순수 연산 시간인지)** 를 먼저 통일해야 정확한 비교가 가능합니다.

</details>